# M1 — Triage dermatológico asistido por IA
### SI4006 · Proyecto Integrador · Universidad EAFIT

Sistema de triage dermatológico asistido por IA: clasifica la descripción de síntomas de un
paciente en **urgente** o **no urgente**, para apoyar la decisión de un enfermero/a de
triage sobre si referir con prioridad a dermatología.

| Integrante | Módulo a cargo |
|---|---|
| Isabella Idárraga | Dataset y preprocesamiento |
| María Alejandra Ocampo | Modelo de texto (M1) y métricas |
| Camilo Salazar | Componente visual (M4) |
| Juan Esteban Alzate | Integración del sistema y ética |

Este notebook es la entrega de **M1**: reproduce el pipeline completo (justificación del
modelo, dataset, fine-tuning con LoRA, evaluación) y deja evidencia de que el modelo
entrenado carga desde cero y produce salidas coherentes.

In [1]:
import os

# Robusto sin importar si el kernel arranca en notebooks/ o en la raíz del repo.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Directorio de trabajo:", os.getcwd())

Directorio de trabajo: C:\Users\juane\OneDrive\Documentos\EAFIT\2026-02\Topicos IA\Proyecto\triage-dermatologico-ia


## 1 · Dominio, usuario y tarea

**Dominio:** triage dermatológico en atención primaria o telemedicina — pacientes que
consultan por una lesión o cambio en la piel, describiendo sus síntomas.

**Usuario y decisión:** un enfermero/a o médico/a general que atiende la primera consulta
(triage), antes de que el caso llegue a un dermatólogo. La decisión que cambia es si el caso
se refiere con urgencia a dermatología o se agenda como cita de rutina, en vez de una misma
fila de espera por orden de llegada.

**Tarea del modelo (M1):** clasificación binaria de urgencia a partir de la descripción
libre del paciente: **urgente** (posible melanoma u otra lesión de riesgo) vs. **no urgente**
(control de rutina). Se eligió esta simplificación binaria sobre las 7 categorías originales
del dataset porque refleja mejor la pregunta real que un enfermero de triage (no
dermatólogo) necesita responder.

**Métrica de éxito definida por el equipo** (`docs/Plantilla_Proyecto_Integrador_Salud.pdf`):
Recall ≥ 0.85 en la clase "urgente" (priorizar no dejar pasar casos graves, aunque se
sacrifique algo de precisión) y F1 macro > 0.75.

## 2 · Selección y justificación del modelo base

Según la regla de la Sesión 3 (clasificar/extraer → familia *encoder*), se descartaron
modelos decoder y encoder-decoder. Se compararon dos candidatos encoder:

- `distilbert-base-multilingual-cased` — ~135M parámetros, +100 idiomas, licencia Apache 2.0
- `dccuchile/bert-base-spanish-wwm-cased` (**BETO**) — ~110M parámetros, entrenado solo en
  español, licencia CC BY 4.0

**Evidencia empírica** (`scripts/comparar_tokenizadores.py`, corrido sobre
`data/corpus_dominio_seed.csv`): se tokenizó el corpus de dominio con ambos modelos y se
comparó cuánto fragmentan los términos clínicos clave. Resultados cargados abajo.

In [2]:
import pandas as pd

beto = pd.read_csv("results/tokenizacion_beto-español.csv")
distilbert = pd.read_csv("results/tokenizacion_distilbert-multilingual.csv")

print(f"Promedio tokens/palabra — BETO (español):         {beto['tokens_por_palabra'].mean():.2f}")
print(f"Promedio tokens/palabra — distilbert-multilingual: {distilbert['tokens_por_palabra'].mean():.2f}")
print("\nMenor promedio = menos fragmentación = mejor para el dominio.")

beto.head()

Promedio tokens/palabra — BETO (español):         1.43
Promedio tokens/palabra — distilbert-multilingual: 1.60

Menor promedio = menos fragmentación = mejor para el dominio.


,texto,n_palabras,n_tokens,tokens_por_palabra
0,"Mujer, 44 años, con antecedentes de exéresis d...",53,97,1.83
1,Tengo un lunar en la espalda que ha cambiado d...,26,28,1.08
2,Tengo un lunar en el abdomen que ha cambiado d...,25,27,1.08
3,Lesión pigmentada de bordes ligeramente verruc...,30,64,2.13
4,Tengo una placa áspera y descamativa en la pie...,21,28,1.33


**Observaciones:** BETO fragmenta menos en promedio sobre el corpus completo, y gana
claramente en términos como "melanoma" (2 piezas vs. 3). No es un barrido total —
`distilbert-multilingual` fragmenta menos "dermatofibroma" y "nevus"— pero la fila que
decide la comparación es el promedio sobre el corpus completo, no los términos sueltos.

**Decisión:** `dccuchile/bert-base-spanish-wwm-cased` (BETO), por su menor fragmentación
general de términos clínicos en español y su menor tamaño (110M vs. 135M parámetros) — en
línea con la idea de "el modelo correcto para el dominio, no el más grande" (lección de
Chinchilla vista en clase).

**Licencia:** BETO se distribuye bajo CC BY 4.0, compatible con el uso académico de este
proyecto.

## 3 · Dataset: construcción y documentación

**136 ejemplos** en `data/corpus_final_M1.csv`, combinando dos fuentes:

- **10 casos reales**, extraídos de SPACCC y CodiEsp (corpus clínicos públicos en español,
  CC BY 4.0), filtrados por código CIE-10 (C43, C44, D22, D23, L57, L82) y verificados uno
  por uno a mano para descartar falsos positivos. Redactados en lenguaje de reporte clínico.
- **126 casos sintéticos**, generados por plantillas (`scripts/generador_corpus_sintetico.py`)
  basadas en los mismos criterios clínicos que los casos reales (regla ABCDE para melanoma,
  aspecto perlado de carcinoma basocelular, textura áspera de queratosis...). Imitan el
  lenguaje coloquial de un paciente — son los que realmente representan la distribución de
  entrada que el sistema recibirá en producción.

7 categorías estilo HAM10000 (`mel`, `akiec`, `bcc`, `bkl`, `nv`, `df`, `vasc`), recodificadas
a 2 clases de urgencia. Split estratificado por categoría, con semilla fija (42).

In [3]:
corpus = pd.read_csv("data/corpus_final_M1.csv")
print(f"Total: {len(corpus)} ejemplos\n")

print("Por split:")
print(corpus["split"].value_counts(), "\n")

print("Por categoría (HAM10000):")
print(corpus["categoria_ham10000"].value_counts(), "\n")

print("Por tipo de fuente:")
print(corpus["tipo_fuente"].value_counts(), "\n")

print("Por etiqueta de urgencia:")
print(corpus["etiqueta_urgencia"].value_counts())

Total: 136 ejemplos

Por split:
split
train    93
test     26
val      17
Name: count, dtype: int64 

Por categoría (HAM10000):
categoria_ham10000
bcc      22
vasc     20
akiec    20
df       19
mel      19
nv       18
bkl      18
Name: count, dtype: int64 

Por tipo de fuente:
tipo_fuente
sintetico    126
real          10
Name: count, dtype: int64 

Por etiqueta de urgencia:
etiqueta_urgencia
no_urgente    75
urgente       61
Name: count, dtype: int64


**Limitaciones conocidas** (documentadas en `data/README.md`):

- No hay casos reales de `nv` ni `bkl` en ninguno de los dos corpus — dependen 100% del
  corpus sintético.
- Los 10 casos reales están en lenguaje de reporte clínico, mientras los 126 sintéticos
  imitan lenguaje de paciente — solo estos últimos representan el tipo de texto que el
  sistema recibirá en producción.
- Las plantillas sintéticas repiten estructura de oración (solo cambian ubicación anatómica
  y tiempo de evolución), lo que favorece sobreajuste a la forma de la plantilla en vez del
  criterio clínico — esto se observa empíricamente más abajo, en la sección de evaluación.
- 136 ejemplos es un corpus pequeño para fine-tuning profundo desde cero; es razonable en
  esta etapa porque BETO ya viene preentrenado en español general y solo necesita afinar una
  frontera de decisión binaria.

## 4 · Implementación del fine-tuning con LoRA

Se ajusta BETO con LoRA (librería `peft`) sobre las capas de atención `query`/`value`, tarea
`SEQ_CLS` — clasificación de secuencias, no generación de texto (a diferencia del
laboratorio S04, que usaba un modelo decoder). El pipeline completo vive en
`scripts/train.py`; esta celda lo ejecuta tal cual correría desde la terminal, para que el
entrenamiento sea reproducible desde este mismo notebook.

In [4]:
%pip install -q peft accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import time


def esperar_archivo_estable(path, espera=0.5, intentos=20):
    """Espera a que un archivo deje de cambiar de tamaño.

    Este repo vive en una carpeta sincronizada con OneDrive: se observó que, al
    correr entrenamiento y evaluación como subprocesos consecutivos (celdas de
    abajo), OneDrive puede introducir un pequeño retraso entre que un proceso
    termina de escribir un archivo y el archivo queda disponible para lectura,
    causando que la siguiente celda lea una versión vieja o incompleta.
    """
    anterior = -1
    for _ in range(intentos):
        actual = os.path.getsize(path) if os.path.exists(path) else -1
        if actual == anterior and actual > 0:
            return
        anterior = actual
        time.sleep(espera)

In [6]:
!python scripts/train.py

Dispositivo: cpu
Train: 93  |  Val: 17  |  Test: 26
trainable params: 296,450 || all params: 110,148,868 || trainable%: 0.2691
{'loss': '0.6242', 'grad_norm': '1.62', 'learning_rate': '0.0001956', 'epoch': '0.4167'}
{'loss': '0.79', 'grad_norm': '5.407', 'learning_rate': '0.00019', 'epoch': '0.8333'}
{'eval_loss': '0.7272', 'eval_accuracy': '0.5294', 'eval_f1_macro': '0.3462', 'eval_recall_urgente': '0', 'eval_runtime': '0.7058', 'eval_samples_per_second': '24.09', 'eval_steps_per_second': '4.251', 'epoch': '1'}
{'loss': '0.7427', 'grad_norm': '1.42', 'learning_rate': '0.0001844', 'epoch': '1.25'}
{'loss': '0.7196', 'grad_norm': '1.916', 'learning_rate': '0.0001789', 'epoch': '1.667'}
{'eval_loss': '0.671', 'eval_accuracy': '0.4706', 'eval_f1_macro': '0.32', 'eval_recall_urgente': '0', 'eval_runtime': '0.7325', 'eval_samples_per_second': '23.21', 'eval_steps_per_second': '4.095', 'epoch': '2'}
{'loss': '0.6566', 'grad_norm': '1.432', 'learning_rate': '0.0001733', 'epoch': '2.083'}
{'lo


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15707.51it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differ

In [7]:
esperar_archivo_estable("models/lora-triage/adapter_model.safetensors")

Los hiperparámetros quedan registrados automáticamente en `results/lora_config.json` (esto
es lo que pide la rúbrica como "hiperparámetros registrados", en un archivo explícito, no
solo escondido dentro del código):

In [8]:
import json

with open("results/lora_config.json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

{
  "modelo_base": "dccuchile/bert-base-spanish-wwm-cased",
  "semilla": 42,
  "lora_r": 8,
  "lora_alpha": 16,
  "lora_dropout": 0.05,
  "lora_target_modules": [
    "query",
    "value"
  ],
  "max_length": 128,
  "learning_rate": 0.0002,
  "num_epochs": 15,
  "batch_size": 8,
  "n_train": 93,
  "n_val": 17,
  "n_test": 26
}


## 5 · Baseline y reporte de métricas

**Baseline:** como BETO para clasificación se inicializa con una cabeza de clasificación
aleatoria, "BETO sin entrenar" no sería una comparación justa — mediría ruido, no un modelo
real. Se usa en su lugar el **baseline de mayoría**: predice siempre la clase más frecuente
del set de entrenamiento (`no_urgente`). Es la comparación estándar en clasificación.

`scripts/evaluate.py` calcula ambos sobre el mismo test set (26 ejemplos, nunca vistos en
entrenamiento) y guarda `results/baseline_metrics.json` y `results/lora_metrics.json`.

In [9]:
!python scripts/evaluate.py

Evaluando sobre el set de TEST (26 ejemplos, nunca vistos en entrenamiento)

Clase mayoritaria en train: no_urgente

=== BASELINE (predice siempre la clase mayoritaria) ===
  Accuracy:        0.5769
  F1 macro:        0.3659
  Recall urgente:  0.0000
  Falsos negativos (urgente clasificado como no_urgente): 11

=== MODELO CON LORA (BETO fine-tuneado) ===
  Accuracy:        0.6154
  F1 macro:        0.4583
  Recall urgente:  0.0909
  Falsos negativos (urgente clasificado como no_urgente): 10

COMPARACIÓN BASELINE vs. LORA
  accuracy              baseline=0.5769  lora=0.6154  delta=+0.0385
  f1_macro              baseline=0.3659  lora=0.4583  delta=+0.0925
  recall_urgente        baseline=0.0000  lora=0.0909  delta=+0.0909

Guardado en results/baseline_metrics.json y results/lora_metrics.json



Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13406.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differ

In [10]:
esperar_archivo_estable("results/lora_metrics.json")

In [11]:
with open("results/baseline_metrics.json", encoding="utf-8") as f:
    baseline = json.load(f)
with open("results/lora_metrics.json", encoding="utf-8") as f:
    lora = json.load(f)

comparacion = pd.DataFrame({
    "baseline": {k: baseline[k] for k in ["accuracy", "f1_macro", "recall_urgente"]},
    "lora":     {k: lora[k]     for k in ["accuracy", "f1_macro", "recall_urgente"]},
})
comparacion["delta"] = comparacion["lora"] - comparacion["baseline"]
comparacion.round(4)

,baseline,lora,delta
accuracy,0.5769,0.6154,0.0385
f1_macro,0.3659,0.4583,0.0925
recall_urgente,0.0000,0.0909,0.0909


In [12]:
META_RECALL = 0.85
META_F1 = 0.75

print("Lectura honesta del delta (esta corrida):\n")
print(f"  recall_urgente:  lora={lora['recall_urgente']:.4f}  "
      f"(meta del proyecto: >= {META_RECALL})  "
      f"{'CUMPLE' if lora['recall_urgente'] >= META_RECALL else 'NO cumple'}")
print(f"  f1_macro:        lora={lora['f1_macro']:.4f}  "
      f"(meta del proyecto: > {META_F1})  "
      f"{'CUMPLE' if lora['f1_macro'] > META_F1 else 'NO cumple'}")
print(f"  vs. baseline de mayoría: accuracy {'+' if comparacion.loc['accuracy','delta']>=0 else ''}"
      f"{comparacion.loc['accuracy','delta']:.4f}, "
      f"f1_macro {'+' if comparacion.loc['f1_macro','delta']>=0 else ''}"
      f"{comparacion.loc['f1_macro','delta']:.4f}, "
      f"recall_urgente {'+' if comparacion.loc['recall_urgente','delta']>=0 else ''}"
      f"{comparacion.loc['recall_urgente','delta']:.4f}")

cm = lora["matriz_confusion"]
print(f"\n  Falsos negativos (urgente -> no_urgente): {cm['urgente_predicho_como_no_urgente']} "
      f"de {cm['urgente_predicho_como_no_urgente'] + cm['urgente_predicho_correctamente']} casos urgentes")
print(f"  Falsos positivos (no_urgente -> urgente): {cm['no_urgente_predicho_como_urgente']} "
      f"de {cm['no_urgente_predicho_como_urgente'] + cm['no_urgente_predicho_correctamente']} casos no urgentes")

Lectura honesta del delta (esta corrida):

  recall_urgente:  lora=0.0909  (meta del proyecto: >= 0.85)  NO cumple
  f1_macro:        lora=0.4583  (meta del proyecto: > 0.75)  NO cumple
  vs. baseline de mayoría: accuracy +0.0385, f1_macro +0.0925, recall_urgente +0.0909

  Falsos negativos (urgente -> no_urgente): 10 de 11 casos urgentes
  Falsos positivos (no_urgente -> urgente): 0 de 15 casos no urgentes


**Dónde mejoró:** en toda corrida observada durante el desarrollo, el modelo con LoRA supera
al baseline de mayoría — el baseline nunca detecta un caso urgente por construcción (siempre
predice la clase mayoritaria), mientras que LoRA sí aprende señal real del corpus.

**Dónde no mejoró / limitaciones — y por qué:**

- **La meta técnica del proyecto** (Recall ≥ 0.85 en "urgente", F1 macro > 0.75, definidas en
  `docs/Plantilla_Proyecto_Integrador_Salud.pdf`) **no se cumple de forma consistente.** En la
  corrida documentada oficialmente en `docs/comparacion_resultados.md` (la que queda commiteada
  en `results/` y `models/lora-triage/` del repo), el recall en urgentes fue 0.45 — 6 falsos
  negativos de 11 casos urgentes, el error clínico más grave posible en este sistema — y F1
  macro 0.66, por debajo de 0.75.
- **Durante el desarrollo se intentó mitigar el sobreajuste** reduciendo las épocas de 15 a 10,
  siguiendo la advertencia del material de la Sesión 4 sobre memorización con pocos datos. El
  recall subió levemente pero la precisión colapsó (más del doble de falsos positivos) y
  F1/accuracy empeoraron — no fue una mejora neta, así que se descartó. Detalle completo en
  `docs/comparacion_resultados.md`.
- **Se aprendió algo más al construir este notebook:** con solo 93 ejemplos de entrenamiento y
  11 casos urgentes en test, **reentrenar con la misma configuración produce resultados muy
  distintos entre corridas** — durante las pruebas de este notebook se observó recall_urgente
  oscilando entre 0.0 y 1.0 en corridas separadas, sin cambiar nada del código. Esto no es un
  error del pipeline: es la consecuencia directa de medir sobre 11 casos urgentes de test (cada
  uno pesa ~9% de la métrica) y entrenar sobre un corpus pequeño y poco variado. Por eso el
  repo fija una corrida como el resultado oficial documentado, en vez de reportar "lo que salió
  esta vez".

**Conclusión:** LoRA aprende señal real del corpus, pero el sistema todavía no cumple el
criterio de seguridad clínica que el propio equipo definió, y el resultado exacto es sensible
al azar del entrenamiento dado el tamaño del dataset. La causa de fondo es el tamaño y la poca
variabilidad lingüística del corpus sintético, no el modelo base ni la técnica de fine-tuning.
La siguiente fase del proyecto contempla ampliar el corpus con mayor diversidad de redacción,
lo cual debería reducir esta varianza además de subir el recall.

## 6 · Evidencia: el modelo carga desde cero y produce salidas coherentes

Celda obligatoria (criterio 3 de la rúbrica): se recarga el adaptador LoRA guardado en
`models/lora-triage/` **desde cero** — reimportando todo con prefijo `_` y sin reutilizar
ningún objeto en memoria de las celdas anteriores — y se prueba sobre un ejemplo nuevo, que
no está en el corpus de entrenamiento ni de test.

In [13]:
esperar_archivo_estable("models/lora-triage/adapter_model.safetensors")

In [14]:
# Reimport limpio: no reutiliza tokenizer/model de celdas anteriores.
import torch as _torch
from transformers import AutoModelForSequenceClassification as _AutoModel, AutoTokenizer as _AutoTok
from peft import PeftModel as _PeftModel

_LORA_DIR = "models/lora-triage"
_MODEL_ID = "dccuchile/bert-base-spanish-wwm-cased"
_LABEL2ID = {"no_urgente": 0, "urgente": 1}
_ID2LABEL = {v: k for k, v in _LABEL2ID.items()}

_tokenizer = _AutoTok.from_pretrained(_LORA_DIR)
_base_model = _AutoModel.from_pretrained(
    _MODEL_ID, num_labels=2, id2label=_ID2LABEL, label2id=_LABEL2ID
)
_modelo_recargado = _PeftModel.from_pretrained(_base_model, _LORA_DIR)
_modelo_recargado.eval()

# Ejemplo nuevo, no presente en el corpus
_ejemplo_nuevo = (
    "Me salió una mancha oscura en la planta del pie hace dos meses, ha ido creciendo "
    "y ahora tiene bordes irregulares y varios tonos de color."
)

_inputs = _tokenizer(_ejemplo_nuevo, return_tensors="pt", truncation=True, max_length=128)
with _torch.no_grad():
    _logits = _modelo_recargado(**_inputs).logits
_pred = _torch.argmax(_logits, dim=-1).item()
_probs = _torch.softmax(_logits, dim=-1)[0]

print("Texto:          ", _ejemplo_nuevo)
print("Predicción:     ", _ID2LABEL[_pred])
print(f"Probabilidades:  no_urgente={_probs[0]:.3f}  urgente={_probs[1]:.3f}")

C:\Users\juane\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18984.42it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Texto:           Me salió una mancha oscura en la planta del pie hace dos meses, ha ido creciendo y ahora tiene bordes irregulares y varios tonos de color.
Predicción:      urgente
Probabilidades:  no_urgente=0.358  urgente=0.642


---

Con esto, M1 queda completo: modelo base justificado con evidencia empírica del tokenizador,
dataset documentado con splits reproducibles, fine-tuning con LoRA reproducible desde este
mismo notebook, baseline y métricas comparadas con una lectura honesta del delta —incluyendo
lo que no funcionó—, y evidencia explícita de que el modelo guardado carga desde cero y
responde de forma coherente sobre un ejemplo nuevo.